# 10 · Checkpointing, fault tolerance & the budget at scale

An unattended loop must survive preemption. Ray Train persists a checkpoint each
epoch and restarts from it on failure; the budget becomes a hard deadline that
scores the *best checkpoint so far*. We demo checkpoint → restore locally.

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from harness.data import make_synthetic_dataset, DatasetSpec, load_split, class_names

DATA = Path("../data/bdd-tiny.lance")
if not DATA.exists():
    make_synthetic_dataset(DATA, DatasetSpec(n=3000, seed=7))
print("dataset:", DATA, "| NOTE: these chapters scale to bdd-small/full; here we")
print("demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.")

dataset: ../data/bdd-tiny.lance | NOTE: these chapters scale to bdd-small/full; here we
demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.


In [2]:
try:
    import ray, tempfile, os, torch
    from ray.train import ScalingConfig, Checkpoint
    from ray.train.torch import TorchTrainer
    HAVE_RAY = True
except Exception as e:
    HAVE_RAY = False; print("Ray not installed -> code only:", e)

2026-05-25 16:03:24,806	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-05-25 16:03:26,622	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-05-25 16:03:26,947	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
def train_func(cfg):
    import tempfile, os, torch, torch.nn.functional as F, ray.train
    from harness.data import load_split, class_names
    from harness.model import build_model
    x, y, _ = load_split(cfg["data"], "train")
    model = build_model(cfg, len(class_names()))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    xt = torch.from_numpy(x.transpose(0, 3, 1, 2)).float(); yt = torch.from_numpy(y).long()
    best = 1e9
    for epoch in range(cfg["epochs"]):
        opt.zero_grad(); loss = F.cross_entropy(model(xt), yt); loss.backward(); opt.step()
        with tempfile.TemporaryDirectory() as d:
            torch.save(model.state_dict(), os.path.join(d, "m.pt"))
            ray.train.report({"loss": float(loss)},
                             checkpoint=Checkpoint.from_directory(d))

if HAVE_RAY:
    if not ray.is_initialized():
        ray.init(num_cpus=4, logging_level="ERROR", include_dashboard=False)
    trainer = TorchTrainer(
        train_func,
        train_loop_config={"data": str(DATA.resolve()), "epochs": 4,
                           "model": {"name": "tiny_cnn", "width": 16, "depth": 2}},
        scaling_config=ScalingConfig(num_workers=1))
    result = trainer.fit()
    print("best checkpoint:", result.checkpoint is not None,
          "| final loss:", round(result.metrics["loss"], 4))
    print("on preemption Ray restarts from this checkpoint; the budget deadline")
    print("scores whatever checkpoint exists -> safe truncation, no lost run.")
else:
    print("(install ray[train] to run this)")

(TrainController pid=5079) Requesting resources: {'CPU': 1} * 1


(TrainController pid=5079) Attempting to start training worker group of size 1 with the following resources: [{'CPU': 1}] * 1


(PlacementGroupCleaner pid=5164) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


(RayTrainWorker pid=5168) Setting up process group for: env:// [rank=0, world_size=1]
(RayTrainWorker pid=5168) [W525 16:03:38.577580253 socket.cpp:764] [c10d] The client socket cannot be initialized to connect to [::ffff:192.0.2.2]:49977 (errno: 97 - Address family not supported by protocol).
(TrainController pid=5079) Started training worker group of size 1: 
(TrainController pid=5079) - (ip=192.0.2.2, pid=5168) world_rank=0, local_rank=0, node_rank=0
(PlacementGroupCleaner pid=5164) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


(PlacementGroupCleaner pid=5164) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


...

(PlacementGroupCleaner pid=5164) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


(PlacementGroupCleaner pid=5164) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


(PlacementGroupCleaner pid=5164) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


(PlacementGroupCleaner pid=5164) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


best checkpoint: True | final loss: 0.7633
on preemption Ray restarts from this checkpoint; the budget deadline
scores whatever checkpoint exists -> safe truncation, no lost run.


Because each iteration writes to its own run dir and the *decision* derives only
from `results.tsv` + the immutable evaluator, a crashed iteration can be retried
without corrupting history — and fixed seeds make the retry reproducible.